# 3.5 Lab - Exploding Join (Skew, Explosion, Spill)
**Lessons:** Skew · Spill

A nightly job joins `orders` to `customer_addresses` to report revenue per city. It got slower every week and now sometimes fails. You will find and fix three classic problems:
1. **Join explosion**: a non-unique join key multiplies rows
2. **Skew**: one huge key makes one task run far longer than the rest
3. **Spill**: a partition too big for memory spills to disk

Replace each `<FILL_IN>` and run the CHECK cells. The solution is in `03-performance-optimization/solutions/`.

In [ ]:
from adelab.classroom import setup
from adelab import perf
from pyspark.sql import functions as F, Window

DA = setup("3.5", reset=True)
spark.conf.set("spark.sql.shuffle.partitions", 32)

# 2M orders. Customer 0 is a marketplace "guest checkout" account that owns 30% of all orders.
(spark.range(2_000_000)
      .withColumn("customer_id", F.when(F.rand(seed=1) < 0.3, F.lit(0))
                                  .otherwise((F.rand(seed=2) * 100_000).cast("int") + 1))
      .withColumn("amount", F.round(F.rand(seed=3) * 200, 2))
      .write.saveAsTable("orders"))

# Address HISTORY: 1-5 rows per customer, and 20 rows for customer 0. Only one row per customer is current.
addr = (spark.range(0, 100_001).withColumnRenamed("id", "customer_id")
        .withColumn("versions", F.when(F.col("customer_id") == 0, 20).otherwise((F.col("customer_id") % 5) + 1))
        .withColumn("v", F.explode(F.sequence(F.lit(1), F.col("versions"))))
        .withColumn("city", F.element_at(F.array(*[F.lit(c) for c in ["Berlin", "Paris", "Madrid", "Rome", "Vienna", "Oslo"]]),
                                         ((F.col("customer_id") + F.col("v")) % 6 + 1).cast("int")))
        .withColumn("is_current", F.col("v") == F.col("versions"))
        .select(F.col("customer_id").cast("int").alias("customer_id"), "city", "v", "is_current"))
addr.write.saveAsTable("customer_addresses")

# Force a shuffle join so you can see the problems (the fix in step 5 brings broadcasting back).
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")

## Step 1 - Run the job as written

In [ ]:
naive = (spark.table("orders").join(spark.table("customer_addresses"), "customer_id")
         .groupBy("city").agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue")))
r_naive = perf.run(naive, "naive join")
display(perf.compare(r_naive))
display(perf.stage_summary(spark, last=3))

Two red flags:
- `join_output_rows` is **far larger** than the 2 M orders. The join multiplies rows (and revenue!), so the report is **wrong**, not just slow.
- In the join stage, `max_task_ms` is many times `median_task_ms`. That is **skew**.

![A key with 3 orders and 3 address versions produces 3 x 3 = 9 join rows; keeping only the current address gives one row per order](../images/3.5-join-explosion.svg)

**How to read it:** every row on the left meets every row with the same key on the right. Customer 0 has 600 k orders and 20 address
versions, hence the 12 M rows in `join_output_rows` above. The Fix box is what you build in Step 3.

## Step 2 - Diagnose: key distribution on both sides
Compute the top 5 `customer_id`s by row count for `orders` **and** for `customer_addresses`.

In [ ]:
top_orders = <FILL_IN>
top_addr = <FILL_IN>
display(top_orders)
display(top_addr)

In [ ]:
# CHECK step 2
assert top_orders.first()["customer_id"] == 0 and top_orders.first()["count"] > 500_000
assert top_addr.first()["customer_id"] == 0 and top_addr.first()["count"] == 20
print("Step 2 passed: customer 0 is hot on both sides -> 600k x 20 = 12M rows from ONE key")

## Step 3 - Fix the explosion: join to exactly one row per customer
The report needs the **current** address. Make the right side unique on the join key.

In [ ]:
current_addr = <FILL_IN>
fixed = (spark.table("orders").join(current_addr, "customer_id")
         .groupBy("city").agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue")))
r_fixed = perf.run(fixed, "join to current address")
display(perf.compare(r_naive, r_fixed))
perf.plot([r_naive, r_fixed], "join_output_rows", "Rows produced by the join", highlight="join to current address",
          note="one row per order");

The fixed join should stop at exactly 2,000,000 rows, one per order; everything above that in the naive bar is duplication.

In [ ]:
# CHECK step 3
assert current_addr.groupBy("customer_id").count().filter("count > 1").count() == 0, "right side must be unique"
assert r_fixed["join_output_rows"] == 2_000_000, r_fixed
print("Step 3 passed")

## Step 4 - Fix the skew
The result is now correct, but customer 0 still sends 30% of all orders to **one** reducer. Check the stage summary, then enable **AQE skew-join handling**. It splits oversized partitions into several tasks and replicates the matching rows of the other side. The thresholds are lowered because this table is small.

![With skew, one task holding the hot key runs far longer than the others. AQE skew join splits that partition into several sub-tasks](../images/3.5-skew-aqe-split.svg)

**How to read it:** each bar is one task on a time axis. The stage finishes when its longest bar finishes, so one red bar sets the
runtime. In `stage_summary` that is a high `skew_ratio` (max / median). After AQE, the hot partition runs as several
shorter tasks and the join plan is marked `skew=true` (the `skew_join` column).

In [ ]:
display(perf.stage_summary(spark, last=1))

In [ ]:
spark.conf.set(<FILL_IN>, "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1MB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "1MB")
r_aqe = perf.run(fixed, "current address + AQE skew join")
display(perf.compare(r_fixed, r_aqe))
display(perf.stage_summary(spark, last=1))

In [ ]:
# CHECK step 4
assert r_aqe["skew_join"], "AQE did not mark the join as skewed - is spark.sql.adaptive.skewJoin.enabled true?"
print("Step 4 passed")

### Manual alternative: salting
Before AQE, skew was fixed by **salting**: spread the hot key over N random sub-keys on the big side, and replicate the small side N times. You still need it for skewed **aggregations** and window functions, which AQE does not split.

![Salting: the hot key gets a random salt 0..N-1 on the big side, the small side is copied N times, and the join on (key, salt) runs as N balanced tasks](../images/3.5-salting.svg)

**How to read it:** the join key becomes `(customer_id, salt)`. The 600 k hot rows are spread over N tasks, and each task still finds
its address row because the small side was copied once per salt value. This is the code in the next cell with N = 16.

In [ ]:
N = 16
salted_orders = spark.table("orders").withColumn("salt", (F.rand(seed=9) * N).cast("int"))
salted_addr = current_addr.withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(N - 1))))
salted = (salted_orders.join(salted_addr, ["customer_id", "salt"])
          .groupBy("city").agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue")))
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
display(perf.compare(r_fixed, perf.run(salted, f"salted join (N={N})")))
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

## Step 5 - The best fix: don't shuffle the big side at all
`current_addr` has only ~100 k rows. Re-enable broadcasting, or use a hint, so the 2 M orders are never shuffled for the join. Skew in the join then disappears.

In [ ]:
broadcast_join = <FILL_IN>
r_bc = perf.run(broadcast_join, "broadcast current address")
display(perf.compare(r_naive, r_fixed, r_aqe, r_bc))
perf.plot([r_naive, r_fixed, r_aqe, r_bc], "seconds", "Query time per fix", highlight="broadcast current address",
          note="no shuffle of the big side");

Read the bars top to bottom as the sequence of fixes. Laptop timings are noisy, so trust the direction more than the exact numbers.

In [ ]:
# CHECK step 5
assert r_bc["join_strategy"] == "BroadcastHashJoin", r_bc["join_strategy"]
print("Step 5 passed")

## Step 6 - Spill
A data scientist adds "running total per customer" with a window function. Window functions buffer a **whole partition key** in memory, and customer 0 has 600 k rows. The buffer threshold is lowered here to simulate a small executor. Watch `spill_bytes`:

![A partition larger than execution memory spills the overflow to local disk and reads it back](../images/3.5-spill.svg)

**How to read it:** the top bar is the window buffer for customer 0. The part that fits stays in execution memory; the rest is
serialized to local disk and read back, which the UI reports as *Spill (memory)* / *Spill (disk)* and `perf` as `spill_bytes`.

In [ ]:
spark.conf.set("spark.sql.windowExec.buffer.spill.threshold", "10000")
spark.conf.set("spark.sql.windowExec.buffer.in.memory.threshold", "10000")
w = Window.partitionBy("customer_id").orderBy("id").rowsBetween(Window.unboundedPreceding, 0)
running = spark.table("orders").withColumn("running_total", F.sum("amount").over(w))
r_spill = perf.run(running, "running total per customer (window)")
display(perf.compare(r_spill))
display(perf.stage_summary(spark, last=1))

Spill means data is written to and read back from local disk. It is slow, but not fatal. Fixes, best first:
1. **Change the logic**: does the hot key need the computation? Here, exclude or pre-aggregate customer 0 (the guest account)
2. More, smaller partitions (salting, more shuffle partitions). This does not help when **one key** is the problem
3. More memory per core: a memory-optimized instance type, or fewer cores per executor

Apply fix 1: compute running totals only for real customers (`customer_id <> 0`).

In [ ]:
running_fixed = <FILL_IN>
r_nospill = perf.run(running_fixed, "running total, guest account excluded")
display(perf.compare(r_spill, r_nospill))
perf.plot([r_spill, r_nospill], "spill_bytes", "Spill", highlight="running total, guest account excluded",
          note="hot key excluded");

The spill should drop to (almost) zero once the hot key no longer has to fit in one window buffer.

In [ ]:
# CHECK step 6
assert r_spill.get("spill_bytes", 0) > 0, "expected spill in the original window query"
assert r_nospill.get("spill_bytes", 0) < r_spill.get("spill_bytes", 0)
print("Step 6 passed")

In [ ]:
for k in ("spark.sql.windowExec.buffer.spill.threshold", "spark.sql.windowExec.buffer.in.memory.threshold",
          "spark.sql.autoBroadcastJoinThreshold", "spark.sql.adaptive.autoBroadcastJoinThreshold"):
    spark.conf.unset(k)
DA.cleanup()